# Glass Ternary Plot Demo

This notebook demonstrates the refactored `glass_ternary_plot` functionality that now accepts:
- 3 element arguments in the form Al-Fe-Ni
- Minimum values for aaxis, baxis, and caxis
- Custom ternary plot layout with colored axes

## Updated Function Signature

```python
def create_ternary_diagram(df, elements=None, a_min=0, b_min=0, c_min=0):
```

### Parameters:
- `df`: DataFrame with 3-element compositions
- `elements`: List of 3 element symbols in order [A, B, C] (e.g., ['Al', 'Fe', 'Ni'])
- `a_min`: Minimum value for A-axis (0-1 range, where 0.4 = 40%)
- `b_min`: Minimum value for B-axis (0-1 range)
- `c_min`: Minimum value for C-axis (0-1 range)

## Key Features

### 1. Custom Element Order
You can specify exactly which elements go on which axes:
- A-axis (red): First element in the list
- B-axis (blue): Second element in the list  
- C-axis (green): Third element in the list

### 2. Custom Axis Minimums
Set minimum values for each axis (0-1 range):
- `a_min=0.4` means A-axis starts at 40%
- `b_min=0.1` means B-axis starts at 10%
- `c_min=0.0` means C-axis starts at 0%

### 3. Ternary Plot Layout
The plot uses the exact layout specification:
- Red colored A-axis with grid lines
- Blue colored B-axis with grid lines
- Green colored C-axis with grid lines
- Percentage tick format
- Colored glass-forming ability (blue=glass, red=non-glass)

### 4. Interactive Features
- Hover over points to see composition details
- Automatic title generation with element names and axis minimums
- Responsive layout


In [1]:
# Import required libraries
import sys
from pathlib import Path

# Import the functions from our glass_ternary_diag_functions module
from glass_ternary_diag_functions import (
    load_glass_data, 
    filter_three_element_compositions,
    create_ternary_diagram,
    create_custom_ternary_plot
)

print("Libraries imported successfully!")


Libraries imported successfully!


## Example 1: Basic Usage with Element Order

Create a ternary plot with Al-Ni-Fe in that specific order:


In [2]:
# Example 1: Basic ternary plot with Al-Ni-Fe order
fig1 = create_custom_ternary_plot(
    elements=['Al', 'Ni', 'Fe']
)

if fig1:
    print("✓ Basic Al-Ni-Fe ternary plot created successfully!")
    fig1.show()
else:
    print("✗ Failed to create basic ternary plot")


Filtered to compositions containing combination: Al-Ni-Fe
Found 77 compositions for Al-Ni-Fe
✓ Basic Al-Ni-Fe ternary plot created successfully!


## Example 2: Custom Axis Minimums

Create a ternary plot with Al ≥ 40%, Fe ≥ 0%, Ni ≥ 0%:


## Example 5: Grid of Ternary Plots

Create a 3-column grid of ternary plots for a specific list of element combinations:


In [6]:
# Example 5: Grid of ternary plots for specific combinations
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Define the combinations we want to plot
combinations = ['Al-Ni-Fe', 'Al-Ni-Zr', 'B-Fe-Ni', 'Si-Ti-Zr']

# Load the data
data_file = Path("data_files/database/glass.json")
if data_file.exists():
    df = load_glass_data(data_file)
    
    # Create subplots with 2 rows and 2 columns
    fig = make_subplots(
        rows=2, cols=2,
        specs=[[{"type": "ternary"}, {"type": "ternary"}],
               [{"type": "ternary"}, {"type": "ternary"}]],
        subplot_titles=[f"{combo}" for combo in combinations],
        horizontal_spacing=0.1,
        vertical_spacing=0.15
    )
    
    # Plot each combination
    for i, combination in enumerate(combinations):
        row = (i // 2) + 1
        col = (i % 2) + 1
        
        # Filter data for this specific combination using the function
        combo_data = filter_three_element_compositions(df, target_combination=combination)
        
        if len(combo_data) > 0:
            # Get the three elements from the combination
            elements = combination.split('-')
            
            # Separate glass-forming and non-glass-forming points
            glass_data = combo_data[combo_data['forms_glass'] == True]
            non_glass_data = combo_data[combo_data['forms_glass'] == False]
            
            # Add glass-forming points (blue)
            if not glass_data.empty:
                fig.add_trace(
                    go.Scatterternary(
                        a=glass_data[elements[0]],
                        b=glass_data[elements[1]], 
                        c=glass_data[elements[2]],
                        mode='markers',
                        marker=dict(color='blue', size=6, symbol='circle'),
                        name='Glass' if i == 0 else '',  # Only show legend for first plot
                        showlegend=(i == 0),
                        text=glass_data['composition'],
                        hovertemplate=f"<b>%{{text}}</b><br>{elements[0]}: %{{a}}%<br>{elements[1]}: %{{b}}%<br>{elements[2]}: %{{c}}%<br><extra></extra>"
                    ),
                    row=row, col=col
                )
            
            # Add non-glass-forming points (red)
            if not non_glass_data.empty:
                fig.add_trace(
                    go.Scatterternary(
                        a=non_glass_data[elements[0]],
                        b=non_glass_data[elements[1]],
                        c=non_glass_data[elements[2]],
                        mode='markers',
                        marker=dict(color='red', size=6, symbol='circle'),
                        name='Non-Glass' if i == 0 else '',  # Only show legend for first plot
                        showlegend=(i == 0),
                        text=non_glass_data['composition'],
                        hovertemplate=f"<b>%{{text}}</b><br>{elements[0]}: %{{a}}%<br>{elements[1]}: %{{b}}%<br>{elements[2]}: %{{c}}%<br><extra></extra>"
                    ),
                    row=row, col=col
                )
            
            # Update ternary subplot layout
            fig.update_layout({
                f"ternary{i+1}": dict(
                    aaxis=dict(
                        title=dict(text=f"Component {elements[0]}", font=dict(color="red")),
                        gridcolor="red", linecolor="red", tickfont=dict(color="red"),
                        tickformat=".0%", ticks="outside", showgrid=True, showline=True, min=0
                    ),
                    baxis=dict(
                        title=dict(text=f"Component {elements[1]}", font=dict(color="blue")),
                        gridcolor="blue", linecolor="blue", tickfont=dict(color="blue"),
                        tickformat=".0%", ticks="outside", showgrid=True, showline=True, min=0
                    ),
                    caxis=dict(
                        title=dict(text=f"Component {elements[2]}", font=dict(color="green")),
                        gridcolor="green", linecolor="green", tickfont=dict(color="green"),
                        tickformat=".0%", ticks="outside", showgrid=True, showline=True, min=0
                    ),
                    sum=1
                )
            })
            
            print(f"✓ {combination}: {len(combo_data)} compositions ({len(glass_data)} glass-forming)")
        else:
            print(f"✗ {combination}: No compositions found")
    
    # Update overall layout
    fig.update_layout(
        title="Ternary Plot Grid: Multiple Element Combinations",
        title_x=0.5,
        height=800,
        showlegend=True,
        font=dict(size=12),
        margin=dict(b=50, l=50, r=50, t=80)
    )
    
    # Count valid combinations
    valid_combinations = 0
    for combination in combinations:
        combo_data = filter_three_element_compositions(df, target_combination=combination)
        if len(combo_data) > 0:
            valid_combinations += 1
    
    print(f"\n✓ Grid plot created with {valid_combinations} valid combinations")
    fig.show()
    
else:
    print("Data file not found")


Filtered to compositions containing combination: Al-Ni-Fe
✓ Al-Ni-Fe: 77 compositions (20 glass-forming)
Filtered to compositions containing combination: Al-Ni-Zr
✓ Al-Ni-Zr: 178 compositions (149 glass-forming)
Filtered to compositions containing combination: B-Fe-Ni
✓ B-Fe-Ni: 126 compositions (94 glass-forming)
Filtered to compositions containing combination: Si-Ti-Zr
✓ Si-Ti-Zr: 79 compositions (75 glass-forming)
Filtered to compositions containing combination: Al-Ni-Fe
Filtered to compositions containing combination: Al-Ni-Zr
Filtered to compositions containing combination: B-Fe-Ni
Filtered to compositions containing combination: Si-Ti-Zr

✓ Grid plot created with 4 valid combinations


In [4]:
# Example 2: Ternary plot with Al ≥ 40%
fig2 = create_custom_ternary_plot(
    elements=['Al', 'Fe', 'Ni'],
    a_min=0.4,  # Al axis starts at 40%
    b_min=0.0,  # Fe axis starts at 0%
    c_min=0.0   # Ni axis starts at 0%
)

if fig2:
    print("✓ Al ≥ 40% ternary plot created successfully!")
    fig2.show()
else:
    print("✗ Failed to create Al ≥ 40% ternary plot")


Filtered to compositions containing combination: Al-Fe-Ni
Found 77 compositions for Al-Fe-Ni
✓ Al ≥ 40% ternary plot created successfully!


### Examples Covered:
1. **Basic Usage**: Simple ternary plot with custom element order
2. **Custom Axis Minimums**: Setting minimum values for each axis
3. **Different Element Order**: Changing the order of elements on axes
4. **Direct Function Usage**: Using the core functions directly with pre-loaded data
5. **Grid of Plots**: Multiple ternary plots in a single figure for comparison


### Additional Benefits:
- **Grid Visualization**: Compare multiple element combinations side-by-side in a single plot


## Example 3: Different Element Order

Create a ternary plot with Ni-Fe-Al order:


In [ ]:
# Example 3: Ternary plot with Ni-Fe-Al order
fig3 = create_custom_ternary_plot(
    elements=['Ni', 'Fe', 'Al'],
    a_min=0.3,  # Ni axis starts at 30%
    b_min=0.1,  # Fe axis starts at 10%
    c_min=0.0   # Al axis starts at 0%
)

if fig3:
    print("✓ Ni-Fe-Al ternary plot created successfully!")
    fig3.show()
else:
    print("✗ Failed to create Ni-Fe-Al ternary plot")


## Example 4: Direct Function Usage

You can also use the `create_ternary_diagram` function directly with pre-loaded data:


In [ ]:
# Example 4: Direct function usage with custom parameters
print("Example 4: Direct function usage with Al ≥ 50%...")

# Load the data first
data_file = Path("data_files/database/glass.json")

if data_file.exists():
    print("Loading glass data...")
    df = load_glass_data(data_file)
    print(f"Loaded {len(df)} compositions")
    
    # Filter to Al-Fe-Ni compositions
    df_3elem = filter_three_element_compositions(df, target_combination='Al-Fe-Ni')
    print(f"Found {len(df_3elem)} Al-Fe-Ni compositions")
    
    if len(df_3elem) > 0:
        # Create ternary plot with custom parameters
        fig4 = create_ternary_diagram(
            df_3elem,
            elements=['Al', 'Fe', 'Ni'],
            a_min=0.5,  # Al ≥ 50%
            b_min=0.0,  # Fe ≥ 0%
            c_min=0.0   # Ni ≥ 0%
        )
        
        if fig4:
            print("✓ Direct usage ternary plot created successfully!")
            fig4.show()
        else:
            print("✗ Failed to create direct usage ternary plot")
    else:
        print("No Al-Fe-Ni compositions found")
else:
    print(f"Data file not found: {data_file}")
    print("Please ensure the glass.json file exists in the data_files/database/ directory")


## Summary

The ternary plots are displayed directly in the notebook cells above. Each plot is interactive and includes:

- **Hover functionality**: Hover over points to see composition details
- **Zoom and pan**: Use mouse wheel to zoom, click and drag to pan
- **Legend**: Toggle visibility of glass-forming vs non-glass-forming points
- **Responsive layout**: Plots adjust to notebook cell width

All plots use the exact layout specification with colored axes and custom minimums as specified.

### Key Benefits:
1. **Custom Element Order**: Specify exactly which elements go on which axes
2. **Custom Axis Minimums**: Set minimum values for each axis (0-1 range)
3. **Consistent Styling**: All plots use the same axis colors and layout
4. **Interactive Exploration**: Hover over any point for detailed composition information
5. **Easy Comparison**: All plots use the same Al, Ni, X ordering for direct comparison
